# 🏥 Sports Injury Prediction using Machine Learning

An end-to-end machine learning project for predicting athlete injury risk.

In [ ]:
# ============================================================
# SPORTS INJURY PREDICTION USING MACHINE LEARNING
# ============================================================
# An end-to-end Machine Learning project for predicting
# whether an athlete is at high risk of sustaining an injury.
#
# Workflow:
# 1. Generate Synthetic Dataset
# 2. Exploratory Data Analysis
# 3. Data Preprocessing
# 4. Train-Test Split
# 5. Dummy Baseline Model
# 6. Train Multiple ML Models
# 7. 5-Fold Cross Validation
# 8. Model Evaluation
# 9. Feature Importance
# 10. Save Best Model
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    RocCurveDisplay
)


warnings.filterwarnings("ignore")

RANDOM_STATE = 42


print("=" * 70)
print("SPORTS INJURY PREDICTION USING MACHINE LEARNING")
print("=" * 70)

print("\nAll libraries imported successfully!")


# ============================================================
# 2. CREATE PROJECT DIRECTORIES
# ============================================================

os.makedirs("models", exist_ok=True)
os.makedirs("preprocessing", exist_ok=True)

print("Project directories are ready!")


# ============================================================
# 3. GENERATE SYNTHETIC DATASET
# ============================================================

def generate_sports_injury_dataset(
    n_samples=1000,
    random_state=42
):
    """
    Generate a synthetic dataset for sports injury risk prediction.

    Parameters
    ----------
    n_samples : int
        Number of athlete records to generate.

    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    pandas.DataFrame
        Synthetic sports injury dataset.
    """

    rng = np.random.default_rng(random_state)

    # Athlete characteristics

    age = rng.integers(
        18,
        40,
        n_samples
    )

    training_load = rng.uniform(
        1,
        10,
        n_samples
    )

    previous_injuries = rng.poisson(
        1.5,
        n_samples
    )

    sleep_quality = rng.uniform(
        1,
        10,
        n_samples
    )

    nutrition_score = rng.uniform(
        1,
        10,
        n_samples
    )

    muscle_fatigue = rng.uniform(
        1,
        10,
        n_samples
    )

    joint_flexibility = rng.uniform(
        1,
        10,
        n_samples
    )

    hydration_level = rng.uniform(
        1,
        10,
        n_samples
    )

    playing_surface = rng.choice(
        ["Grass", "Hard", "Synthetic"],
        n_samples
    )

    # Create a synthetic injury risk score.
    # Higher training load, previous injuries, and fatigue
    # increase risk, while recovery-related factors reduce risk.

    risk_score = (
        0.30 * training_load
        + 0.70 * previous_injuries
        + 0.40 * muscle_fatigue
        - 0.30 * sleep_quality
        - 0.20 * nutrition_score
        - 0.25 * hydration_level
        - 0.20 * joint_flexibility
        + rng.normal(0, 2, n_samples)
    )

    # Convert the continuous score into a binary target.

    threshold = np.median(risk_score)

    injury_risk = (
        risk_score > threshold
    ).astype(int)

    # Create DataFrame.

    df = pd.DataFrame({
        "Age": age,
        "Training_Load": training_load,
        "Previous_Injuries": previous_injuries,
        "Sleep_Quality": sleep_quality,
        "Nutrition_Score": nutrition_score,
        "Muscle_Fatigue": muscle_fatigue,
        "Joint_Flexibility": joint_flexibility,
        "Hydration_Level": hydration_level,
        "Playing_Surface": playing_surface,
        "Injury_Risk": injury_risk
    })

    return df


# ============================================================
# 4. CREATE DATASET
# ============================================================

df = generate_sports_injury_dataset(
    n_samples=1000,
    random_state=RANDOM_STATE
)

print("\nDataset generated successfully!")

print("\nDataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
print(df.head())


# ============================================================
# 5. EXPLORATORY DATA ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 70)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
print(df.describe())


# ============================================================
# 6. INJURY RISK DISTRIBUTION
# ============================================================

risk_counts = (
    df["Injury_Risk"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(7, 5))

plt.bar(
    ["Low Risk", "High Risk"],
    risk_counts.values
)

plt.title("Injury Risk Distribution")
plt.xlabel("Injury Risk")
plt.ylabel("Number of Athletes")

plt.tight_layout()
plt.show()


# ============================================================
# 7. CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(10, 7))

correlation_matrix = (
    df.select_dtypes(include=np.number)
    .corr()
)

plt.imshow(
    correlation_matrix,
    aspect="auto"
)

plt.colorbar()

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns
)

for i in range(len(correlation_matrix.columns)):
    for j in range(len(correlation_matrix.columns)):
        plt.text(
            j,
            i,
            f"{correlation_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

plt.title("Feature Correlation Heatmap")

plt.tight_layout()
plt.show()


# ============================================================
# 8. DATA PREPROCESSING
# ============================================================

print("\n" + "=" * 70)
print("DATA PREPROCESSING")
print("=" * 70)


# One-hot encode categorical feature.

df_encoded = pd.get_dummies(
    df,
    columns=["Playing_Surface"],
    drop_first=True
)


# Define features and target.

X = df_encoded.drop(
    "Injury_Risk",
    axis=1
)

y = df_encoded[
    "Injury_Risk"
]


# Ensure numeric compatibility across environments.

X = X.astype(float)


print("\nFeature Shape:")
print(X.shape)

print("\nFeature Names:")
print(list(X.columns))


# Save feature names.

joblib.dump(
    list(X.columns),
    "preprocessing/feature_names.pkl"
)

print("\nFeature names saved successfully!")


# ============================================================
# 9. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\n" + "=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)


# ============================================================
# 10. DUMMY BASELINE MODEL
# ============================================================

print("\n" + "=" * 70)
print("DUMMY BASELINE MODEL")
print("=" * 70)


dummy_model = DummyClassifier(
    strategy="most_frequent"
)

dummy_model.fit(
    X_train,
    y_train
)

dummy_predictions = dummy_model.predict(
    X_test
)

dummy_accuracy = accuracy_score(
    y_test,
    dummy_predictions
)

print(
    f"\nDummy Baseline Accuracy: "
    f"{dummy_accuracy:.4f}"
)


# ============================================================
# 11. DEFINE MACHINE LEARNING MODELS
# ============================================================

models = {

    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        )
    ]),

    "Support Vector Machine": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            SVC(
                probability=True,
                random_state=RANDOM_STATE
            )
        )
    ]),

    "Random Forest": Pipeline([
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE
            )
        )
    ])
}

print("\nMachine Learning models initialized successfully!")


# ============================================================
# 12. 5-FOLD CROSS VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("5-FOLD CROSS VALIDATION")
print("=" * 70)


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}


cv_results = []


for name, model in models.items():

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1 Score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })


cv_results_df = pd.DataFrame(
    cv_results
)

cv_results_df = cv_results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)


print("\nCross Validation Results:")
print(cv_results_df)


# ============================================================
# 13. TRAIN AND EVALUATE MODELS
# ============================================================

print("\n" + "=" * 70)
print("MODEL TRAINING AND EVALUATION")
print("=" * 70)


results = []
trained_models = {}


for name, model in models.items():

    print("\n" + "-" * 50)
    print(name)
    print("-" * 50)


    # Train model.

    model.fit(
        X_train,
        y_train
    )


    # Generate predictions.

    predictions = model.predict(
        X_test
    )

    probabilities = model.predict_proba(
        X_test
    )[:, 1]


    # Calculate evaluation metrics.

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        probabilities
    )


    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")


    # Store results.

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

    trained_models[name] = model


results_df = pd.DataFrame(
    results
)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)


print("\nFinal Model Results:")
print(results_df)


# ============================================================
# 14. MODEL COMPARISON VISUALIZATION
# ============================================================

plt.figure(figsize=(10, 6))

plt.bar(
    results_df["Model"],
    results_df["F1 Score"]
)

plt.title(
    "Model Comparison Based on F1 Score"
)

plt.xlabel(
    "Machine Learning Model"
)

plt.ylabel(
    "F1 Score"
)

plt.ylim(0, 1)

plt.xticks(rotation=15)

plt.tight_layout()
plt.show()


# ============================================================
# 15. CONFUSION MATRICES
# ============================================================

fig, axes = plt.subplots(
    1,
    len(trained_models),
    figsize=(18, 5)
)

for ax, (name, model) in zip(
    axes,
    trained_models.items()
):

    predictions = model.predict(
        X_test
    )

    cm = confusion_matrix(
        y_test,
        predictions
    )

    image = ax.imshow(
        cm
    )

    ax.set_title(name)

    ax.set_xlabel(
        "Predicted"
    )

    ax.set_ylabel(
        "Actual"
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(
        ["Low Risk", "High Risk"]
    )

    ax.set_yticklabels(
        ["Low Risk", "High Risk"]
    )

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center"
            )

    fig.colorbar(
        image,
        ax=ax
    )

plt.tight_layout()
plt.show()


# ============================================================
# 16. ROC CURVE COMPARISON
# ============================================================

plt.figure(figsize=(8, 6))

for name, model in trained_models.items():

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    RocCurveDisplay.from_predictions(
        y_test,
        probabilities,
        name=name
    )

plt.title(
    "ROC Curve Comparison"
)

plt.tight_layout()
plt.show()


# ============================================================
# 17. FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("FEATURE IMPORTANCE")
print("=" * 70)


rf_pipeline = trained_models[
    "Random Forest"
]

random_forest = rf_pipeline.named_steps[
    "model"
]


feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest.feature_importances_
})


feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)


print("\nFeature Importance:")
print(feature_importance)


# ============================================================
# 18. FEATURE IMPORTANCE VISUALIZATION
# ============================================================

plt.figure(figsize=(10, 6))

plt.barh(
    feature_importance["Feature"],
    feature_importance["Importance"]
)

plt.title(
    "Feature Importance for Injury Risk Prediction"
)

plt.xlabel(
    "Importance"
)

plt.ylabel(
    "Feature"
)

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


# ============================================================
# 19. SELECT BEST MODEL
# ============================================================

best_model_name = results_df.loc[
    results_df["F1 Score"].idxmax(),
    "Model"
]

best_model = trained_models[
    best_model_name
]

best_model_f1 = results_df.loc[
    results_df["F1 Score"].idxmax(),
    "F1 Score"
]


print("\n" + "=" * 70)
print(
    f"BEST MODEL: {best_model_name}"
)
print(
    f"BEST TEST F1 SCORE: {best_model_f1:.4f}"
)
print("=" * 70)


# ============================================================
# 20. SAVE BEST MODEL
# ============================================================

model_path = (
    "models/injury_risk_pipeline.pkl"
)

joblib.dump(
    best_model,
    model_path
)

print(
    f"\nBest model saved successfully: {model_path}"
)


# ============================================================
# 21. FINAL PROJECT SUMMARY
# ============================================================

best_cv_model = cv_results_df.loc[
    cv_results_df["F1 Score"].idxmax(),
    "Model"
]

best_cv_f1 = cv_results_df.loc[
    cv_results_df["F1 Score"].idxmax(),
    "F1 Score"
]


print("\n" + "=" * 70)
print(
    "SPORTS INJURY PREDICTION - FINAL SUMMARY"
)
print("=" * 70)

print(f"\nDataset Size: {df.shape}")

print(
    f"\nDummy Baseline Accuracy: "
    f"{dummy_accuracy:.4f}"
)

print(
    f"\nBest Test Model: "
    f"{best_model_name}"
)

print(
    f"Best Test F1 Score: "
    f"{best_model_f1:.4f}"
)

print(
    f"\nBest Cross Validation Model: "
    f"{best_cv_model}"
)

print(
    f"Best Cross Validation F1 Score: "
    f"{best_cv_f1:.4f}"
)

print(
    f"\nSaved Model: {model_path}"
)

print(
    "\nSaved Feature Names: "
    "preprocessing/feature_names.pkl"
)

print("\n" + "=" * 70)
print("PROJECT EXECUTED SUCCESSFULLY!")
print("=" * 70)
